In [ ]:
%load_ext autoreload
%autoreload 2

%load_ext rich

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
from pathlib import Path
from typing import ClassVar, Iterable, Literal

import ollama
import pandas as pd
import requests
import yaml
from ollama import ChatResponse
from pydantic import BaseModel, ValidationError, model_validator
from tqdm import tqdm

from aymurai.utils.json_data import load_json, save_json
from aymurai.utils.yaml_data import load_yaml

## Configuración

In [ ]:
MODEL = "gpt-oss:20b"
DEVICE = "cuda"

API_BASE_URL = os.getenv("API_BASE_URL", "https://aymurai.collectiveai.io/api")
ENDPOINT = f"{API_BASE_URL}/api/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "/resources/data/restricted/defensoria/pdfs")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "30"))

PROMPT_CONFIG_PATH = Path("/resources/llm/defensoria_extractor.yml")
RESULTS_PATH = Path(f"./information-extractio-results-{DEVICE}.json")

print(f"Model:           {MODEL}")
print(f"Prompt config:   {PROMPT_CONFIG_PATH}")
print(f"Results output:  {RESULTS_PATH}")
print(f"Target endpoint: {ENDPOINT}")

## System prompt desde YAML

In [ ]:
config = load_yaml(PROMPT_CONFIG_PATH)
config.keys()

In [ ]:
def build_fields_block(fields: dict) -> str:
    """Convierte el dict de campos del YAML a un bloque Markdown para el prompt."""
    lines = []
    for field, description in fields.items():
        desc = str(description).strip().replace("\n", " ")
        lines.append(f"- **{field}**: {desc}")
    return "\n".join(lines)


def build_system_prompt(config: dict) -> str:
    prompts = config["system-prompts"]
    fields = config["fields"]
    taxonomy = config["taxonomy"]
    schema = config["output-format"]["schema"]

    schema_json = json.dumps(schema, ensure_ascii=False, indent=2)

    return f"""{prompts["information-extraction"]}

{prompts["extraction-guidelines"]}
# Campos a extraer

{build_fields_block(fields)}

# Taxonomía de temas y subtemas

{taxonomy}

# Formato de salida (JSON)

Responde exclusivamente con un objeto JSON válido con la siguiente estructura:

```json
{schema_json}
```
"""


system_prompt = build_system_prompt(config)
print(system_prompt)

## Modelos Pydantic

In [ ]:
# Produces: {tema: frozenset(subtemas)}
_taxonomy: dict[str, frozenset[str]] = {
    tema: frozenset(subtemas)
    for entry in yaml.safe_load(config["taxonomy"])
    for tema, subtemas in entry.items()
}

Tema = Literal[tuple(_taxonomy)]


class Destinatario(BaseModel):
    nombre: str | None = None
    cargo: str | None = None
    destinatario_principal: bool


class DataExtraction(BaseModel):
    numero_recomendacion: str | None = None
    fecha_recomendacion: str | None = None
    destinatarios: list[Destinatario]
    tema: Tema | None = None
    subtema: str | None = None
    destinatarios_por_sector: list[str]
    datos_personales: bool
    contenido_para_publicar: str

    taxonomy: ClassVar[dict[str, frozenset[str]]] = _taxonomy

    @model_validator(mode="after")
    def validate_tema_subtema(self) -> DataExtraction:
        if self.subtema is None:
            return self

        if self.tema is None:
            raise ValueError("No se puede especificar un subtema sin un tema")

        valid_subtemas = self.taxonomy[self.tema]

        if self.subtema not in valid_subtemas:
            raise ValueError(
                f"El subtema {self.subtema!r} no pertenece al tema "
                f"{self.tema!r}. Subtemas válidos: {sorted(valid_subtemas)}"
            )

        return self


DataExtraction.model_json_schema()

## Carga de documentos

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {"file": (file_path.name, file_path.open("rb"), mime_type)}

    try:
        start = time.perf_counter()
        response = session.post(ENDPOINT, files=files, timeout=REQUEST_TIMEOUT)
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

In [ ]:
# Test rápido con un solo documento
doc_path = documents[-1]
extracted_document = call_extraction_api(requests.Session(), doc_path)["detail"]
document_text = "\n".join(extracted_document["document"])
print(f"Documento: {doc_path.name}")
print(f"Caracteres: {len(document_text)}")
print(document_text[:500])

## Inferencia

In [ ]:
def parse_json_object(raw_output: str) -> dict:
    """Extrae el primer objeto JSON válido de la respuesta del modelo."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_output.strip())
    decoder = json.JSONDecoder()

    for start, char in enumerate(cleaned):
        if char != "{":
            continue
        try:
            parsed, _ = decoder.raw_decode(cleaned[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return parsed

    raise ValueError("No valid JSON object found in model response")


def get_chat_response(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    options: dict = {"num_ctx": 32_768, "num_predict": 8192},
) -> ChatResponse:
    return ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options=options,
    )


def build_validation_retry_prompt(
    system_prompt: str,
    validation_error: str,
    previous_output: str,
) -> str:
    return f"""{system_prompt}

# Corrección obligatoria

Tu respuesta anterior no pasó la validación automática.
Devuelve exclusivamente un objeto JSON válido, sin Markdown ni texto adicional.

Error de validación:
{validation_error}

Respuesta anterior:
{previous_output}
"""


def evaluate_chat_response(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    options: dict = {"num_ctx": 32_768, "num_predict": 8192},
    validation_model: type[BaseModel] | None = None,
    max_retries: int = 2,
) -> dict:
    """Llama al modelo, registra métricas y valida contra un modelo Pydantic.

    Reintenta hasta max_retries veces enviando el error de validación al modelo.
    """
    attempts = []
    current_system_prompt = system_prompt

    for attempt_idx in range(max_retries + 1):
        start = time.time()
        response = get_chat_response(
            system_prompt=current_system_prompt,
            user_prompt=user_prompt,
            model=model,
            options=options,
        )
        end = time.time()

        input_tokens = getattr(response, "prompt_eval_count", None)
        output_tokens = getattr(response, "eval_count", None)
        total_tokens = (
            input_tokens + output_tokens
            if input_tokens is not None and output_tokens is not None
            else None
        )
        input_duration_ms = (
            response.prompt_eval_duration / 1_000_000
            if hasattr(response, "prompt_eval_duration")
            else None
        )
        output_duration_ms = (
            response.eval_duration / 1_000_000
            if hasattr(response, "eval_duration")
            else None
        )
        model_total_duration_ms = (
            response.total_duration / 1_000_000
            if hasattr(response, "total_duration")
            else None
        )
        tokens_per_second = (
            output_tokens / (output_duration_ms / 1000)
            if output_tokens and output_duration_ms
            else None
        )

        attempt_result = {
            "attempt": attempt_idx + 1,
            "chat_response": response.message.content,
            "input_tokens": input_tokens,
            "input_duration_ms": input_duration_ms,
            "output_tokens": output_tokens,
            "output_duration_ms": output_duration_ms,
            "total_tokens": total_tokens,
            "model_duration_ms": model_total_duration_ms,
            "measured_duration_ms": (end - start) * 1000,
            "tokens_per_second": tokens_per_second,
            "raw_response": response.model_dump(),
        }

        if validation_model is None:
            attempts.append(attempt_result)
            break

        try:
            parsed = parse_json_object(response.message.content)
            validated = validation_model.model_validate(parsed)
        except (json.JSONDecodeError, ValidationError, ValueError) as exc:
            last_error = str(exc)
            attempt_result.update(
                {"validation_succeeded": False, "validation_error": last_error}
            )
            attempts.append(attempt_result)
            if attempt_idx >= max_retries:
                break
            current_system_prompt = build_validation_retry_prompt(
                system_prompt=system_prompt,
                validation_error=last_error,
                previous_output=response.message.content,
            )
            continue

        attempt_result.update(
            {
                "validation_succeeded": True,
                "validation_error": None,
                "parsed_response": parsed,
                "validated_response": validated.model_dump(),
            }
        )
        attempts.append(attempt_result)
        break

    final = attempts[-1]
    return {
        "model": model,
        "options": options,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        **{
            k: final[k]
            for k in [
                "chat_response",
                "input_tokens",
                "input_duration_ms",
                "output_tokens",
                "output_duration_ms",
                "total_tokens",
                "model_duration_ms",
                "measured_duration_ms",
                "tokens_per_second",
                "raw_response",
            ]
        },
        "attempts": attempts,
        "num_attempts": len(attempts),
        "validation_succeeded": final.get("validation_succeeded"),
        "validation_error": final.get("validation_error"),
        "parsed_response": final.get("parsed_response"),
        "validated_response": final.get("validated_response"),
    }

### Prueba sobre un documento

In [ ]:
test_result = evaluate_chat_response(
    system_prompt=system_prompt,
    user_prompt=f"Documento: {doc_path.name}" + "\n\n" + document_text,
    model=MODEL,
    validation_model=DataExtraction,
    max_retries=2,
)

print(f"Validation succeeded: {test_result['validation_succeeded']}")
print(f"Attempts: {test_result['num_attempts']}")
if not test_result["validation_succeeded"]:
    print(f"Validation error: {test_result['validation_error']}")
print(f"Input tokens: {test_result['input_tokens']}")
print(f"Output tokens: {test_result['output_tokens']}")
print(f"Speed (tok/s): {test_result['tokens_per_second']:.1f}")

if test_result["validated_response"]:
    extraction = DataExtraction.model_validate(test_result["validated_response"])
    print("\n--- Extracción ---")
    print(extraction.model_dump_json(indent=2))

### Inspección de destinatarios

In [ ]:
if test_result["validated_response"]:
    extraction = DataExtraction.model_validate(test_result["validated_response"])
    dest_df = pd.DataFrame([d.model_dump() for d in extraction.destinatarios])
    print(f"Documento: {doc_path.name}")
    display(dest_df)

## Evaluación sobre el corpus completo

In [ ]:
errors: list[dict] = []
results: list[dict] = load_json(RESULTS_PATH) if RESULTS_PATH.exists() else []
already_processed = {(r["doc_path"], r["model"], r["device"]) for r in results}
print(f"Resultados ya procesados: {len(results)}")

In [ ]:
for doc in tqdm(documents, desc="Extracting information"):
    doc_name = doc.name

    if (doc_name, MODEL, DEVICE) in already_processed:
        tqdm.write(f"Skip (already done): {doc_name}")
        continue

    session = requests.Session()
    extracted = call_extraction_api(session, doc)
    session.close()

    if extracted["status"] != "success":
        tqdm.write(f"Extraction API error for {doc_name}: {extracted['detail']}")
        errors.append({"doc_path": doc_name, "error": extracted["detail"]})
        continue

    text = "\n".join(extracted["detail"]["document"])
    if not text.strip():
        tqdm.write(f"Skip (empty): {doc_name}")
        continue

    tqdm.write(f"Evaluating: {doc_name}")

    try:
        result = evaluate_chat_response(
            system_prompt=system_prompt,
            user_prompt=f"Documento: {doc_name}" + "\n\n" + text,
            model=MODEL,
            options={"num_ctx": 32_768, "num_predict": 8192},
            validation_model=DataExtraction,
            max_retries=2,
        )
        result["doc_path"] = doc_name
        result["device"] = DEVICE

        if not result["validation_succeeded"]:
            tqdm.write(
                f"Validation failed for {doc_name} after {result['num_attempts']} attempts"
            )
            errors.append(
                {
                    "doc_path": doc_name,
                    "model": MODEL,
                    "device": DEVICE,
                    "error": result["validation_error"],
                }
            )
            continue

        results.append(result)
        already_processed.add((doc_name, MODEL, DEVICE))
        save_json(results, RESULTS_PATH)

    except Exception as exc:
        tqdm.write(f"Error evaluating {doc_name}: {exc}")
        errors.append(
            {"doc_path": doc_name, "model": MODEL, "device": DEVICE, "error": str(exc)}
        )

print(f"\nCompletado: {len(results)} resultados, {len(errors)} errores.")

## Análisis de resultados

In [ ]:
raw_results = load_json(RESULTS_PATH)
print(f"Total resultados: {len(raw_results)}")

parsed_rows = []
parse_errors = []

for idx, result in enumerate(raw_results):
    try:
        if result.get("validated_response"):
            extraction = result["validated_response"]
        else:
            extraction = DataExtraction.model_validate(
                parse_json_object(result["chat_response"])
            ).model_dump()

        parsed_rows.append(
            {
                "result_idx": idx,
                "doc_path": result.get("doc_path"),
                "model": result.get("model"),
                "device": result.get("device"),
                "input_tokens": result.get("input_tokens"),
                "output_tokens": result.get("output_tokens"),
                "tokens_per_second": result.get("tokens_per_second"),
                "num_attempts": result.get("num_attempts"),
                **extraction,
            }
        )
    except Exception as exc:
        parse_errors.append({"result_idx": idx, "error": str(exc)})

results_df = pd.DataFrame(parsed_rows)
print(f"Parseados: {len(results_df)} | Errores: {len(parse_errors)}")
results_df.head()

In [ ]:
# Distribución por tema
results_df["tema"].value_counts(dropna=False).to_frame("count")

In [ ]:
# Estadísticas de destinatarios: proporción principal vs notificado por documento
def destinatarios_stats(row) -> dict:
    dests = row["destinatarios"]
    if not isinstance(dests, list):
        return {"n_total": 0, "n_principal": 0, "n_notificacion": 0}
    n_principal = sum(1 for d in dests if d.get("destinatario_principal"))
    return {
        "n_total": len(dests),
        "n_principal": n_principal,
        "n_notificacion": len(dests) - n_principal,
    }


dest_stats = results_df.apply(destinatarios_stats, axis=1, result_type="expand")
display(dest_stats.describe())
results_df[["doc_path"]].join(dest_stats).sort_values("n_total", ascending=False).head(
    10
)

In [ ]:
# Métricas de performance
perf_cols = ["input_tokens", "output_tokens", "tokens_per_second", "num_attempts"]
results_df[perf_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])